In [1]:
import random
import multiprocessing
import pandas as pd
import os
import torch

NUM_GPUS=0

try:
    if torch.cuda.is_available():  
        device = torch.device("cuda")
        NUM_GPUS=torch.cuda.device_count()
        print('There are %d GPU(s) available.' % NUM_GPUS)
        print('We will use the GPU:', torch.cuda.get_device_name())# If not...
    else:
        print('No GPU available, using the CPU instead.')
        device = torch.device("cpu")  
except:
    print('Cuda error using CPU instead.')
    device = torch.device("cpu")  
    
print(device)

# device = torch.device("cpu")  
# print(device)

NUM_PROCESSORS=multiprocessing.cpu_count()
print("Cpu count: ",NUM_PROCESSORS)

There are 1 GPU(s) available.
We will use the GPU: NVIDIA A10
cuda
Cpu count:  32


In [2]:
from pathlib import Path

if os.uname()[1].find('gilbreth')==0: ##if not darwin(mac/locallaptop)
    DIR='/scratch/gilbreth/das90/Dataset/'
elif os.uname()[1].find('unimodular')==0:
    DIR='/scratch2/das90/Dataset/'
elif os.uname()[1].find('Siddharthas')==0:
    DIR='/Users/siddharthashankardas/Purdue/Dataset/'  
else:
    DIR='./Dataset/'
    
Path(DIR).mkdir(parents=True, exist_ok=True)

RESULTS_DIR=DIR+'RESULTS/'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print("Data directory: ", DIR)
print("Result directory:", RESULTS_DIR)

Data directory:  /scratch/gilbreth/das90/Dataset/
Result directory: /scratch/gilbreth/das90/Dataset/RESULTS/


## Packages

In [3]:
import torch.nn as nn
import numpy as np
from torch.nn import init
from random import shuffle, randint
import torch.nn.functional as F
from torch_geometric.datasets import Reddit, PPI, Planetoid
from itertools import combinations, combinations_with_replacement
from sklearn.metrics import f1_score, accuracy_score
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt
import sys
from torch_geometric.data import Data
import logging
import time


## Dataset

In [4]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.datasets import Reddit, Reddit2

In [5]:
def get_data():
        
    DATASET_NAME='Reddit' #"Cora", "CiteSeer", "PubMed"

    if DATASET_NAME in ["Cora", "CiteSeer", "PubMed"]:
        dataset = Planetoid(root=DIR+'Planetoid', name=DATASET_NAME, transform=NormalizeFeatures())

    elif DATASET_NAME == "Reddit2":
        dataset = Reddit2(root=DIR+'Reddit2', transform=NormalizeFeatures())

    elif DATASET_NAME == "Reddit":
        dataset = Reddit(root=DIR+'Reddit', transform=NormalizeFeatures())

    else:    
        raise Exception('dataset not found')

    print()
    print(f'Dataset: {dataset}:')
    print('======================')
    print(f'Number of graphs: {len(dataset)}')
    print(f'Number of features: {dataset.num_features}')
    print(f'Number of classes: {dataset.num_classes}')

    data = dataset[0]  # Get the first graph object.

    print()
    print(data)
    print('===========================================================================================================')

    # Gather some statistics about the graph.
    print(f'Number of nodes: {data.num_nodes}')
    print(f'Number of edges: {data.num_edges}')
    print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
    print(f'Number of training nodes: {data.train_mask.sum()}')
    print(f'Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.2f}')
    print(f'Has isolated nodes: {data.has_isolated_nodes()}')
    print(f'Has self-loops: {data.has_self_loops()}')
    print(f'Is undirected: {data.is_undirected()}')
    
    return data, dataset

#data, dataset = get_data()

### Testing 


### GCN model

In [6]:
import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel
from tqdm import tqdm

from torch_geometric.datasets import Reddit
from torch_geometric.loader import NeighborSampler
from torch_geometric.nn import SAGEConv
from torch_geometric.nn import GCNConv

In [7]:
#https://www.arangodb.com/2021/08/a-comprehensive-case-study-of-graphsage-using-pytorchgeometric/

class SAGE(torch.nn.Module):
    def __init__(self, in_channels, out_channels, hidden_channels, num_layers=2):
        super().__init__()
        torch.manual_seed(1234567)
        self.num_layers = num_layers

        self.convs = torch.nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(self.num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.convs.append(SAGEConv(hidden_channels, out_channels))

    def forward(self, x, adjs):
        for i, (edge_index, _, size) in enumerate(adjs):
            x_target = x[:size[1]]  # Target nodes are always placed first.
            x = self.convs[i]((x, x_target), edge_index)
            if i != self.num_layers - 1:
                x = F.relu(x)
                #x = F.dropout(x, p=0.5, training=self.training)
                x = F.dropout(x, p=0.2, training=self.training)
        return x.log_softmax(dim=-1)

    @torch.no_grad()
    def inference(self, x_all, device, subgraph_loader):
        pbar = tqdm(total=x_all.size(0) * self.num_layers)
        pbar.set_description('Evaluating')

        for i in range(self.num_layers):
            xs = []
            for batch_size, n_id, adj in subgraph_loader:
                edge_index, _, size = adj.to(device)
                x = x_all[n_id].to(device)
                x_target = x[:size[1]]
                x = self.convs[i]((x, x_target), edge_index)
                if i != self.num_layers - 1:
                    x = F.relu(x)
                xs.append(x.cpu())

                pbar.update(batch_size)

            x_all = torch.cat(xs, dim=0)

        pbar.close()

        return x_all

In [8]:
def train(model, data, epochs=100, train_neighbors=[25,10]):
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()
    
    print("Train neighbors: ", train_neighbors)
    
    train_idx = data.train_mask.nonzero(as_tuple=False).view(-1)
#     train_loader = NeighborSampler(data.edge_index, node_idx=train_idx,
#                                    sizes=[25, 10], batch_size=1024,
#                                    shuffle=True, num_workers=0)
    
    train_loader = NeighborSampler(data.edge_index, node_idx=train_idx,
                                   sizes=train_neighbors, batch_size=1024,
                                   shuffle=True, num_workers=0)    
    
    subgraph_loader = NeighborSampler(data.edge_index, node_idx=None,
                                          sizes=[-1], batch_size=2048,
                                          shuffle=False, num_workers=6)
    
    x, y = data.x.to(device), data.y.to(device)
    data.train_mask.to(device)
    data.val_mask.to(device)
    data.test_mask.to(device)
    
    best_acc=0    
    for epoch in range(1,epochs+1):
        
        pbar = tqdm(total=train_idx.size(0))
        pbar.set_description(f'Epoch {epoch:02d}')
        
        total_loss = total_correct = 0
        model.train()
        for batch_size, n_id, adjs in train_loader:
            adjs = [adj.to(device) for adj in adjs]

            optimizer.zero_grad()
            out = model(x[n_id], adjs)
            #loss = F.nll_loss(out, y[n_id[:batch_size]])
            loss = criterion(out, y[n_id[:batch_size]])
            loss.backward()
            optimizer.step()
            
            
        
        
            total_loss += float(loss)
            total_correct += int(out.argmax(dim=-1).eq(y[n_id[:batch_size]]).sum())
            pbar.update(batch_size)

        pbar.close()

        loss = total_loss / len(train_loader)
        approx_acc = total_correct / train_idx.size(0)
        
        
        print(f'Epoch: {epoch:03d}, Training Loss: {loss:.4f}, Training Accuracy: {approx_acc:.4f}')
                
        ####EVALUATION
        if epoch>0 and epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                out = model.inference(x, device, subgraph_loader)
            res = out.argmax(dim=-1) == data.y
            train_acc = int(res[data.train_mask].sum()) / int(data.train_mask.sum())
            val_acc = int(res[data.val_mask].sum()) / int(data.val_mask.sum())
            test_acc = int(res[data.test_mask].sum()) / int(data.test_mask.sum())
            
            print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}')
        
            if test_acc>best_acc:
                best_acc=test_acc

    print ("Best Test Accuracy, ",best_acc)
    
    return model


In [9]:
def GSAGEperformance(data, dataset, epochs=20, train_neighbors=[25,10]):
    model = SAGE(dataset.num_features, dataset.num_classes, hidden_channels=256).to(device)        
    print(model)
    
    train(model, data, epochs, train_neighbors)
    
    return

### Main function

In [10]:
# data, dataset = get_data()
# GSAGEperformance(data, dataset, epochs=10)

In [12]:
if __name__ == '__main__':    
    data, dataset = get_data()    
    GSAGEperformance(data, dataset, epochs=25)
    None
    


Dataset: Reddit():
Number of graphs: 1
Number of features: 602
Number of classes: 41

Data(x=[232965, 602], edge_index=[2, 114615892], y=[232965], train_mask=[232965], val_mask=[232965], test_mask=[232965])
Number of nodes: 232965
Number of edges: 114615892
Average node degree: 491.99
Number of training nodes: 153431
Training node label rate: 0.66
Has isolated nodes: False
Has self-loops: False
Is undirected: True
SAGE(
  (convs): ModuleList(
    (0): SAGEConv(602, 256)
    (1): SAGEConv(256, 41)
  )
)
Train neighbors:  [25, 10]


Epoch 01: 100%|██████████| 153431/153431 [00:10<00:00, 14724.03it/s]


Epoch: 001, Training Loss: 3.4360, Training Accuracy: 0.1070


Epoch 02: 100%|██████████| 153431/153431 [00:10<00:00, 14590.96it/s]


Epoch: 002, Training Loss: 3.4290, Training Accuracy: 0.1076


...

Evaluating: 100%|██████████| 465930/465930 [00:15<00:00, 29894.37it/s]


Epoch: 020, Loss: 3.4254, Train: 0.1076, Val: 0.1466, Test: 0.1483


Epoch 21: 100%|██████████| 153431/153431 [00:11<00:00, 13544.78it/s]


Epoch: 021, Training Loss: 3.4256, Training Accuracy: 0.1076


Epoch 22:  63%|██████▎   | 96256/153431 [00:07<00:05, 11285.28it/s]
KeyboardInterrupt

